## **Project Title:**
### **"Adaptive Federated Learning with Parallel Top-K Gradient Compression for Tuberculosis Detection in Non-IID Medical Imaging"**

#### **Group Members:**
` Misbah Khan`
`Wajeeha Mahmood`
`AI-A`


In [ ]:

import os
import copy
import csv
import random
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, Dataset, TensorDataset
from torchvision import datasets, transforms, models
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from collections import defaultdict
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from tqdm import tqdm

warnings.filterwarnings("ignore")


# ──────────────────────────────────────────────
# 1. CONFIGURATION
# ──────────────────────────────────────────────

@dataclass
class Config:
    dataset_name: str = "brain_tumor"
    data_root: str = "./data"
    backbone: str = "resnet50"
    img_size: int = 224
    pretrained: bool = True
    num_clients: int = 10
    num_rounds: int = 30
    local_epochs: int = 3
    local_batch_size: int = 32
    fraction_fit: float = 1.0
    divergence_threshold_init: float = 0.10
    threshold_ema_alpha: float = 0.2
    lr: float = 1e-3
    weight_decay: float = 1e-4
    warmup_rounds: int = 3
    dirichlet_alpha: float = 0.5
    imbalanced_clients: List[int] = field(default_factory=lambda: [2, 5, 8])
    imbalance_ratio: float = 0.1
    label_smoothing: float = 0.1
    use_mixup: bool = False
    train_ratio: float = 0.70
    val_ratio: float = 0.15
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    save_dir: str = "./checkpoints"
    log_every: int = 5


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ──────────────────────────────────────────────
# 2. DATASET PATHS & TRANSFORMS
# ──────────────────────────────────────────────

DATASET_PATHS = {
    "tb_xray": {
        "train": "TB_Chest_Radiography_Database",
        "test":  None,
    },
    "brain_tumor": {
        "train": "Training",
        "test":  "Testing",
    },
    "diabetic_retinopathy": {
        "train": "colored_images",
        "test":  None,
    },
}


def get_transforms(img_size: int, split: str):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]

    if split == "train":
        return transforms.Compose([
            transforms.Resize((img_size + 32, img_size + 32)),
            transforms.RandomCrop(img_size),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
            transforms.RandomRotation(15),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
            transforms.RandomErasing(p=0.2),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])


from torchvision.datasets import ImageFolder
from PIL import Image, UnidentifiedImageError


class SafeImageFolder(ImageFolder):
    def __getitem__(self, index):
        try:
            return super().__getitem__(index)
        except (UnidentifiedImageError, OSError):
            img = Image.new('RGB', (224, 224), (0, 0, 0))
            if self.transform:
                img = self.transform(img)
            return img, 0


def load_full_dataset(cfg: Config) -> Tuple[datasets.ImageFolder, datasets.ImageFolder]:
    info = DATASET_PATHS[cfg.dataset_name]
    train_path = os.path.join(cfg.data_root, info["train"])

    full_train = SafeImageFolder(train_path, transform=get_transforms(cfg.img_size, "train"))

    if info["test"] is not None:
        test_path = os.path.join(cfg.data_root, info["test"])
        test_ds   = SafeImageFolder(test_path, transform=get_transforms(cfg.img_size, "val"))
        return full_train, test_ds
    else:
        n = len(full_train)
        indices = list(range(n))
        labels  = [full_train.targets[i] for i in indices]
        train_idx, test_idx = train_test_split(
            indices, test_size=(1 - cfg.train_ratio - cfg.val_ratio) * 2,
            stratify=labels, random_state=cfg.seed
        )
        test_base    = SafeImageFolder(train_path, transform=get_transforms(cfg.img_size, "val"))
        train_subset = Subset(full_train, train_idx)
        test_subset  = Subset(test_base, test_idx)
        return train_subset, test_subset


# ──────────────────────────────────────────────
# 3. NON-IID DIRICHLET PARTITION
# ──────────────────────────────────────────────

def dirichlet_partition(
    targets: List[int],
    num_clients: int,
    alpha: float,
    imbalanced_clients: List[int],
    imbalance_ratio: float,
    seed: int
) -> Dict[int, List[int]]:
    np.random.seed(seed)
    targets = np.array(targets)
    num_classes = len(np.unique(targets))
    class_indices = {c: np.where(targets == c)[0].tolist() for c in range(num_classes)}

    for c, idx_list in class_indices.items():
        random.shuffle(idx_list)

    client_indices = defaultdict(list)

    for c, idx_list in class_indices.items():
        proportions = np.random.dirichlet([alpha] * num_clients)
        proportions = proportions / proportions.sum()
        splits = (proportions * len(idx_list)).astype(int)
        splits[-1] = len(idx_list) - splits[:-1].sum()

        # ── BUG FIX #2: Guarantee at least 1 sample per client per class ──
        # Without this, Dirichlet rounding can leave some clients with 0
        # samples for a class, causing the SMOTE loader to be empty and
        # crashing torch.cat() with an empty list error.
        for i in range(num_clients):
            if splits[i] == 0 and splits.max() > 1:
                splits[splits.argmax()] -= 1
                splits[i] = 1

        ptr = 0
        for k in range(num_clients):
            client_indices[k].extend(idx_list[ptr: ptr + splits[k]])
            ptr += splits[k]

    # Apply imbalance to specified clients (paper: clients 2, 5, 8)
    for k in imbalanced_clients:
        idx = client_indices[k]
        idx_targets = targets[idx]
        classes, counts = np.unique(idx_targets, return_counts=True)
        if len(classes) < 2:
            continue
        majority = classes[np.argmax(counts)]
        maj_idx = [i for i in idx if targets[i] == majority]
        rest    = [i for i in idx if targets[i] != majority]
        keep_n  = max(1, int(len(maj_idx) * imbalance_ratio))
        client_indices[k] = random.sample(maj_idx, keep_n) + rest

    return dict(client_indices)


def get_targets(dataset) -> List[int]:
    if isinstance(dataset, Subset):
        full_targets = dataset.dataset.targets
        return [full_targets[i] for i in dataset.indices]
    return dataset.targets


# ──────────────────────────────────────────────
# 4. SMOTE FOR IMBALANCED CLIENTS
# ──────────────────────────────────────────────

def apply_smote_to_loader(loader: DataLoader, img_size: int = 224) -> DataLoader:
    try:
        from imblearn.over_sampling import SMOTE
    except ImportError:
        print("WARNING: imbalanced-learn not installed. Skipping SMOTE.")
        print("Run: pip install imbalanced-learn")
        return loader

    all_imgs, all_labels = [], []
    for imgs, labels in loader:
        all_imgs.append(imgs.cpu())
        all_labels.append(labels.cpu())

    # ── BUG FIX #3: Guard against empty loader before torch.cat ──
    # If the Dirichlet split gave this client zero samples (e.g. after
    # drop_last=True removes the only batch), torch.cat crashes on [].
    if len(all_imgs) == 0:
        print("    [SMOTE skipped] Client loader is empty — no data to augment.")
        return loader

    X = torch.cat(all_imgs).numpy()          # (N, C, H, W)
    y = torch.cat(all_labels).numpy()        # (N,)

    n_samples, C, H, W = X.shape
    X_flat = X.reshape(n_samples, -1)        # (N, C*H*W)

    unique_classes, class_counts = np.unique(y, return_counts=True)
    if len(unique_classes) < 2 or n_samples < 6:
        print("    [SMOTE skipped] Insufficient class diversity or sample count.")
        return loader

    min_samples = min(class_counts)
    k_neighbors = min(5, min_samples - 1)
    if k_neighbors < 1:
        print("    [SMOTE skipped] Too few minority samples for k_neighbors>=1.")
        return loader

    sm = SMOTE(random_state=42, k_neighbors=k_neighbors)
    try:
        X_res, y_res = sm.fit_resample(X_flat, y)
    except Exception as e:
        print(f"    [SMOTE failed: {e}] Using original loader.")
        return loader

    X_tensor = torch.tensor(X_res, dtype=torch.float32).reshape(-1, C, H, W)
    y_tensor  = torch.tensor(y_res, dtype=torch.long)

    ds = TensorDataset(X_tensor, y_tensor)
    return DataLoader(ds, batch_size=loader.batch_size or 32, shuffle=True,
                      num_workers=0, drop_last=True)


# ──────────────────────────────────────────────
# 5. MODEL BUILDER
# ──────────────────────────────────────────────

def build_model(backbone: str, num_classes: int, pretrained: bool = True) -> nn.Module:
    if backbone == "resnet50":
        model = models.resnet50(weights="IMAGENET1K_V2" if pretrained else None)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes),
        )
    elif backbone == "vgg16":
        model = models.vgg16(weights="IMAGENET1K_V1" if pretrained else None)
        in_features = model.classifier[6].in_features
        model.classifier[6] = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes),
        )
    else:
        raise ValueError(f"Unknown backbone: {backbone}. Paper uses only resnet50 or vgg16.")

    for param in model.parameters():
        param.requires_grad = True

    return model


def get_model_params(model: nn.Module) -> List[torch.Tensor]:
    return [p.data.clone() for p in model.parameters()]


def set_model_params(model: nn.Module, params: List[torch.Tensor]):
    for p, new_p in zip(model.parameters(), params):
        p.data.copy_(new_p)


# ──────────────────────────────────────────────
# 6. LOSS FUNCTION
# ──────────────────────────────────────────────

class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, smoothing: float = 0.1):
        super().__init__()
        self.smoothing = smoothing

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        n_classes = logits.size(-1)
        log_probs = F.log_softmax(logits, dim=-1)
        with torch.no_grad():
            smooth = torch.full_like(log_probs, self.smoothing / (n_classes - 1))
            smooth.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        return -(smooth * log_probs).sum(dim=-1).mean()


# ──────────────────────────────────────────────
# 7. LOCAL TRAINING
# ──────────────────────────────────────────────

def local_train(
    model: nn.Module,
    loader: DataLoader,
    cfg: Config,
) -> Tuple[List[torch.Tensor], float]:
    model.train()
    criterion = LabelSmoothingCrossEntropy(cfg.label_smoothing).to(cfg.device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay
    )
    scheduler = CosineAnnealingLR(optimizer, T_max=cfg.local_epochs, eta_min=cfg.lr * 0.1)

    total_loss = 0.0
    steps = 0

    for _ in range(cfg.local_epochs):
        for batch in loader:
            imgs, labels = batch
            imgs, labels = imgs.to(cfg.device), labels.to(cfg.device)

            outputs = model(imgs)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            total_loss += loss.item()
            steps += 1

        scheduler.step()

    updated_params = get_model_params(model)
    return updated_params, total_loss / max(steps, 1)


# ──────────────────────────────────────────────
# 8. DIVERGENCE & ADAPTIVE AGGREGATION
# ──────────────────────────────────────────────

def compute_divergence(client_params_list, global_params):
    total_params = sum(p.numel() for p in global_params)
    deltas = []
    for client_params in client_params_list:
        dist = sum(
            torch.norm(cp.float() - gp.float()).item() ** 2
            for cp, gp in zip(client_params, global_params)
        )
        deltas.append((dist ** 0.5) / (total_params ** 0.5))
    return float(np.mean(deltas))


def fedavg_aggregate(
    client_params_list: List[List[torch.Tensor]],
    client_weights: List[float],
) -> List[torch.Tensor]:
    total = sum(client_weights)
    new_params = []
    for layer_idx in range(len(client_params_list[0])):
        agg = sum(
            (w / total) * cp[layer_idx].float()
            for cp, w in zip(client_params_list, client_weights)
        )
        new_params.append(agg)
    return new_params


def fedsgd_aggregate(
    client_params_list: List[List[torch.Tensor]],
    global_params: List[torch.Tensor],
    client_weights: List[float],
    lr: float,
) -> List[torch.Tensor]:
    total = sum(client_weights)
    new_params = []
    for layer_idx in range(len(global_params)):
        weighted_grad = sum(
            (w / total) * (global_params[layer_idx].float() - cp[layer_idx].float())
            for cp, w in zip(client_params_list, client_weights)
        )
        new_params.append(global_params[layer_idx].float() - lr * weighted_grad)
    return new_params


def adaptive_aggregate(
    client_params_list: List[List[torch.Tensor]],
    global_params: List[torch.Tensor],
    client_weights: List[float],
    divergence: float,
    threshold: float,
    lr: float,
) -> Tuple[List[torch.Tensor], str]:
    if divergence > threshold:
        new_params = fedsgd_aggregate(client_params_list, global_params, client_weights, lr)
        return new_params, "FedSGD"
    else:
        new_params = fedavg_aggregate(client_params_list, client_weights)
        return new_params, "FedAvg"


# ──────────────────────────────────────────────
# 9. EVALUATION
# ──────────────────────────────────────────────

@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: str) -> Tuple[float, float]:
    model.eval()
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

    acc = accuracy_score(all_labels, all_preds) * 100
    f1  = f1_score(all_labels, all_preds, average="weighted") * 100
    return acc, f1


@torch.no_grad()
def evaluate_with_report(model: nn.Module, loader: DataLoader,
                          device: str, class_names: List[str]) -> str:
    model.eval()
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        preds = model(imgs).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
    return classification_report(all_labels, all_preds, target_names=class_names)


# ──────────────────────────────────────────────
# 10. MAIN FEDERATED LEARNING LOOP
# ──────────────────────────────────────────────

def federated_learning(cfg: Config):
    set_seed(cfg.seed)
    os.makedirs(cfg.save_dir, exist_ok=True)
    device = cfg.device
    print(f"\n{'='*60}")
    print(f"  Federated Learning | {cfg.backbone.upper()} | {cfg.dataset_name}")
    print(f"  Device: {device} | Clients: {cfg.num_clients} | Rounds: {cfg.num_rounds}")
    print(f"  [Paper-faithful] Full fine-tuning | SMOTE on clients {cfg.imbalanced_clients}")
    print(f"  Dirichlet alpha={cfg.dirichlet_alpha} | tau_init={cfg.divergence_threshold_init}")
    print(f"{'='*60}\n")

    print("Loading dataset...")
    train_dataset, test_dataset = load_full_dataset(cfg)
    num_classes = len(train_dataset.dataset.classes if isinstance(train_dataset, Subset)
                      else train_dataset.classes)
    class_names = (train_dataset.dataset.classes if isinstance(train_dataset, Subset)
                   else train_dataset.classes)
    print(f"Classes ({num_classes}): {class_names}")

    targets = get_targets(train_dataset)
    client_idx_map = dirichlet_partition(
        targets, cfg.num_clients, cfg.dirichlet_alpha,
        cfg.imbalanced_clients, cfg.imbalance_ratio, cfg.seed
    )

    client_train_loaders = {}
    client_val_loaders   = {}
    client_sizes         = {}

    for k, idx_list in client_idx_map.items():
        if len(idx_list) < 4:
            idx_list = idx_list * 4
        val_n   = max(1, int(len(idx_list) * cfg.val_ratio))
        train_n = len(idx_list) - val_n
        train_idx_k = idx_list[:train_n]
        val_idx_k   = idx_list[train_n:]

        if isinstance(train_dataset, Subset):
            base_train_idx = [train_dataset.indices[i] for i in train_idx_k]
            base_val_idx   = [train_dataset.indices[i] for i in val_idx_k]
            train_sub = Subset(train_dataset.dataset, base_train_idx)
            val_base  = datasets.ImageFolder(
                os.path.join(cfg.data_root, DATASET_PATHS[cfg.dataset_name]["train"]),
                transform=get_transforms(cfg.img_size, "val")
            )
            val_sub = Subset(val_base, base_val_idx)
        else:
            train_sub = Subset(train_dataset, train_idx_k)
            val_sub   = Subset(train_dataset, val_idx_k)

        raw_train_loader = DataLoader(
            train_sub, batch_size=cfg.local_batch_size, shuffle=True,
            num_workers=0, pin_memory=(device == "cuda"), drop_last=True
        )

        if k in cfg.imbalanced_clients:
            print(f"  Applying SMOTE to client {k} (imbalanced)...")
            train_loader_k = apply_smote_to_loader(raw_train_loader, cfg.img_size)
        else:
            train_loader_k = raw_train_loader

        client_train_loaders[k] = train_loader_k
        client_val_loaders[k]   = DataLoader(
            val_sub, batch_size=64, shuffle=False,
            num_workers=0, pin_memory=(device == "cuda")
        )
        client_sizes[k] = len(train_idx_k)

    test_loader = DataLoader(
        test_dataset, batch_size=64, shuffle=False,
        num_workers=0, pin_memory=(device == "cuda")
    )

    print(f"\nClient data sizes: {[client_sizes[k] for k in range(cfg.num_clients)]}")
    print(f"Total training samples: {sum(client_sizes.values())}")

    global_model = build_model(cfg.backbone, num_classes, cfg.pretrained).to(device)
    global_params = get_model_params(global_model)

    history = {
        "round": [], "test_acc": [], "test_f1": [],
        "divergence": [], "method": [], "threshold": []
    }
    best_acc  = 0.0
    threshold = cfg.divergence_threshold_init

    for rnd in range(1, cfg.num_rounds + 1):

        warmup_factor = min(rnd / cfg.warmup_rounds, 1.0)
        eff_lr = cfg.lr * warmup_factor

        n_fit    = max(1, int(cfg.num_clients * cfg.fraction_fit))
        selected = random.sample(range(cfg.num_clients), n_fit)

        client_params_list = []
        client_weight_list = []
        client_loss_list   = []

        for k in selected:
            local_model = build_model(cfg.backbone, num_classes, False).to(device)
            set_model_params(local_model, global_params)

            updated_params, train_loss = local_train(
                local_model,
                client_train_loaders[k],
                cfg,
            )

            client_params_list.append(updated_params)
            client_weight_list.append(client_sizes[k])
            client_loss_list.append(train_loss)

        divergence = compute_divergence(client_params_list, global_params)

        new_params, method = adaptive_aggregate(
            client_params_list, global_params,
            client_weight_list, divergence, threshold, eff_lr
        )

        threshold = (1 - cfg.threshold_ema_alpha) * threshold + \
                     cfg.threshold_ema_alpha * divergence

        global_params = [p.clone() for p in new_params]
        set_model_params(global_model, global_params)

        test_acc, test_f1 = evaluate(global_model, test_loader, device)

        history["round"].append(rnd)
        history["test_acc"].append(test_acc)
        history["test_f1"].append(test_f1)
        history["divergence"].append(divergence)
        history["method"].append(method)
        history["threshold"].append(threshold)

        if test_acc > best_acc:
            best_acc = test_acc
            torch.save(global_model.state_dict(),
                       os.path.join(cfg.save_dir, "best_global_model.pt"))

        if rnd % cfg.log_every == 0 or rnd == 1:
            avg_loss = np.mean(client_loss_list)
            print(f"[Round {rnd:3d}/{cfg.num_rounds}]  "
                  f"Acc={test_acc:.2f}%  F1={test_f1:.2f}%  "
                  f"Div={divergence:.4f}  tau={threshold:.4f}  "
                  f"Method={method}  Loss={avg_loss:.4f}  "
                  f"Best={best_acc:.2f}%")

    print(f"\n{'='*60}")
    print(f"  Training complete. Best test accuracy: {best_acc:.2f}%")
    print(f"{'='*60}\n")

    global_model.load_state_dict(
        torch.load(os.path.join(cfg.save_dir, "best_global_model.pt"),
                   map_location=device)
    )
    report = evaluate_with_report(global_model, test_loader, device, class_names)
    print("Classification Report (best model):\n")
    print(report)

    # ── BUG FIX #1: was `return history, fl_model` — fl_model is never defined.
    # The correct variable holding the trained global model is `global_model`.
    return history, global_model


# ──────────────────────────────────────────────
# 11. CENTRALIZED BASELINE (Differential Privacy)
# ──────────────────────────────────────────────

def add_dp_noise(model: nn.Module, sigma: float = 0.5):
    with torch.no_grad():
        for p in model.parameters():
            if p.grad is not None:
                p.grad += torch.randn_like(p.grad) * sigma


def centralized_baseline(cfg: Config, dp_sigma: float = 0.5, epochs: int = 20):
    set_seed(cfg.seed)
    device = cfg.device
    print(f"\n{'='*60}")
    print(f"  Centralized Baseline (DP sigma={dp_sigma}) | {cfg.backbone.upper()}")
    print(f"{'='*60}\n")

    train_dataset, test_dataset = load_full_dataset(cfg)
    num_classes = len(train_dataset.dataset.classes if isinstance(train_dataset, Subset)
                      else train_dataset.classes)

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,
                              num_workers=0, pin_memory=(device == "cuda"))
    test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False,
                              num_workers=0, pin_memory=(device == "cuda"))

    model     = build_model(cfg.backbone, num_classes, cfg.pretrained).to(device)
    criterion = LabelSmoothingCrossEntropy(cfg.label_smoothing).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=cfg.lr * 0.01)

    best_acc = 0.0
    for epoch in range(1, epochs + 1):
        model.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            add_dp_noise(model, dp_sigma)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
        scheduler.step()

        acc, f1 = evaluate(model, test_loader, device)
        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(),
                       os.path.join(cfg.save_dir, "best_centralized_model.pt"))

        if epoch % 5 == 0:
            print(f"  [Epoch {epoch:3d}] Acc={acc:.2f}%  F1={f1:.2f}%  Best={best_acc:.2f}%")

    print(f"\n  Centralized Best Accuracy: {best_acc:.2f}%\n")
    return best_acc


# ──────────────────────────────────────────────
# 12. SUMMARY PRINTER & LOG SAVER
# ──────────────────────────────────────────────

def print_summary(history: dict):
    rounds  = history["round"]
    accs    = history["test_acc"]
    f1s     = history["test_f1"]
    methods = history["method"]

    n_fedavg = methods.count("FedAvg")
    n_fedsgd = methods.count("FedSGD")

    print(f"\n{'─'*50}")
    print(f"  FL Summary")
    print(f"{'─'*50}")
    print(f"  Final  Acc : {accs[-1]:.2f}%")
    print(f"  Best   Acc : {max(accs):.2f}%  (round {rounds[accs.index(max(accs))]})")
    print(f"  Final  F1  : {f1s[-1]:.2f}%")
    print(f"  FedAvg used: {n_fedavg} rounds ({n_fedavg/len(rounds)*100:.0f}%)")
    print(f"  FedSGD used: {n_fedsgd} rounds ({n_fedsgd/len(rounds)*100:.0f}%)")
    print(f"{'─'*50}\n")


def save_logs(history, path="./logs_tb.csv"):
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=history.keys())
        writer.writeheader()
        rows = [dict(zip(history.keys(), vals))
                for vals in zip(*history.values())]
        writer.writerows(rows)
    print(f"Logs saved to {path}")


# ──────────────────────────────────────────────
# 13. ENTRY POINT
# ──────────────────────────────────────────────

def main():
    cfg = Config(
        dataset_name = "tb_xray",
        data_root    = "/content/drive/MyDrive/fl_data",
        save_dir     = "/content/drive/MyDrive/fl_checkpoints",
        backbone     = "resnet50",
        num_clients              = 10,
        num_rounds               = 30,
        local_epochs             = 3,
        lr                       = 1e-3,
        dirichlet_alpha          = 0.5,
        imbalanced_clients       = [2, 5, 8],
        imbalance_ratio          = 0.1,
        divergence_threshold_init= 0.10,
        use_mixup                = False,
    )

    history, fl_model = federated_learning(cfg)

    save_logs(history, "./logs_tb.csv")
    print_summary(history)

    # Optional: centralized_baseline(cfg, dp_sigma=0.5, epochs=20)

    return history, fl_model


if __name__ == "__main__":
    main()